In [ ]:
import os
import sys

from src.nba_scrapping import *
from src.utils import *
from src.config import *

from datetime import datetime

In [ ]:
start_time = datetime.now()

# Retry errors for historical boxscores (safety)

In [ ]:
#retry_seasons = ["2012-13"]

retry_seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2010, 2025)]


for retry_season in retry_seasons:
    print(f"--- Retry des boxscores échoués pour la saison {retry_season} ---")
    
    season_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, retry_season)
    
    retry_failed_boxscores_for_season(season_batch_dir, max_retries=10)


--- Retry des boxscores échoués pour la saison 2011-12 ---
[INFO] Retrying 0 GAME_IDs for season folder data\raw\boxscores\batches\2011-12
✅ Retry process completed.
--- Retry des boxscores échoués pour la saison 2012-13 ---
[INFO] Retrying 0 GAME_IDs for season folder data\raw\boxscores\batches\2012-13
✅ Retry process completed.
--- Retry des boxscores échoués pour la saison 2013-14 ---
[INFO] Retrying 0 GAME_IDs for season folder data\raw\boxscores\batches\2013-14
✅ Retry process completed.
--- Retry des boxscores échoués pour la saison 2014-15 ---
[INFO] Retrying 3 GAME_IDs for season folder data\raw\boxscores\batches\2014-15
[MOVE] data\raw\boxscores\batches\2014-15\traditional\errors_batch_20.txt to data\raw\boxscores\batches\2014-15\traditional\errors_processed
[MOVE] data\raw\boxscores\batches\2014-15\traditional\errors_batch_8.txt to data\raw\boxscores\batches\2014-15\traditional\errors_processed
[MOVE] data\raw\boxscores\batches\2014-15\advanced\errors_batch_20.txt to data\raw

# Saisons ciblées


In [ ]:
#seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2011, 2015)]
seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2010, 2025)]

seasons


['2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17']

# Merge batches and remove duplicate for all endpoints

In [ ]:
for season in seasons:
    print(f"--- Merging boxscores endpoints for {season} ---")
    
    season_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season)
    
    merge_boxscore_batches_for_season(season_batch_dir)


--- Merging boxscores endpoints for 2011-12 ---
[MERGED] traditional saved to data\raw\boxscores\batches\2011-12\merged_batches\merged_traditional.csv (27638 rows)
[MERGED] advanced saved to data\raw\boxscores\batches\2011-12\merged_batches\merged_advanced.csv (27638 rows)
[MERGED] fourfactors saved to data\raw\boxscores\batches\2011-12\merged_batches\merged_fourfactors.csv (27638 rows)
[MERGED] misc saved to data\raw\boxscores\batches\2011-12\merged_batches\merged_misc.csv (27638 rows)
[MERGED] scoring saved to data\raw\boxscores\batches\2011-12\merged_batches\merged_scoring.csv (27638 rows)
[MERGED] usage saved to data\raw\boxscores\batches\2011-12\merged_batches\merged_usage.csv (27638 rows)
--- Merging boxscores endpoints for 2012-13 ---
[MERGED] traditional saved to data\raw\boxscores\batches\2012-13\merged_batches\merged_traditional.csv (33551 rows)
[MERGED] advanced saved to data\raw\boxscores\batches\2012-13\merged_batches\merged_advanced.csv (33542 rows)
[MERGED] fourfactors s

# Merge endpoints csv into one final with all columns


In [ ]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

for season in seasons:
    print(f"--- Merging all endpoints csv into final for {season} ---")
    
    season_merged_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season, "merged_batches")
    
    all_boxscores_merged_filepath = merge_all_boxscore_stats(season_merged_batch_dir, output_filename=f"final_merged_all_boxscores_{run_timestamp}.csv")
    
    all_boxscores_merged_df = pd.read_csv(all_boxscores_merged_filepath, dtype={'gameId': str})
    
    #analyze_redundant_columns(all_boxscores_merged_df)
    

--- Merging all endpoints csv into final for 2011-12 ---
Shape before cleaning: (27638, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (27638, 100)
✅ All endpoints merged into: data\raw\boxscores\batches\2011-12\merged_batches\final_merged_all_boxscores_2025-06-05_14-31-35.csv (27638 rows)
--- Merging all endpoints csv into final for 2012-13 ---
Shape before cleaning: (33542, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (33542, 100)
✅ All endpoints merged into: data\raw\boxscores\batches\2012-13\merged_batches\final_merged_all_boxscores_2025-06-05_14-31-35.csv (33542 rows)
--- Merging all endpoints csv into final for 2013-14 ---
Shape before cleaning: (33671, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (33671, 100)
✅ All endpoints merged into: data\raw\boxscor

# 🧩 Merge all seasons final boxscores into one


In [8]:
print("\n[MERGE] Concatenating all final_merged_all_boxscores_*.csv into one DataFrame")
final_files = []

for season in seasons:
    season_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season, "merged_batches")
    for file in os.listdir(season_dir):
        if file.startswith("final_merged_all_boxscores_") and file.endswith(".csv"):
            final_files.append(os.path.join(season_dir, file))

if not final_files:
    raise FileNotFoundError("❌ Aucun fichier final_merged_all_boxscores_*.csv trouvé.")

merged_df = pd.concat([pd.read_csv(f, dtype={'gameId': str}) for f in final_files], ignore_index=True)

os.makedirs(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, exist_ok=True)
output_path = os.path.join(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, f"all_seasons_boxscores_merged_{run_timestamp}.csv")
merged_df.to_csv(output_path, index=False)
print(f"✅ Sauvegardé dans : {output_path}")


[MERGE] Concatenating all final_merged_all_boxscores_*.csv into one DataFrame
✅ Sauvegardé dans : data\raw_last\batches_merged\all_seasons_boxscores_merged_2025-06-05_14-31-35.csv


# --- Merge games from each season (take most recent per season) ---


In [18]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

all_games = []

#use custom format because of the way we store the games by periods or not for scrapping
season_dirs = [
    "2011-12_2011-12",
    "2012-13_2014-15",
    "2015-16_2019-20",
    "2020-21_2024-25"
]

for season_dir in season_dirs:
    season_games_dir = os.path.join(DATA_GAMES_DIR, season_dir)
    if os.path.exists(season_games_dir):
        game_files = [os.path.join(season_games_dir, f) for f in os.listdir(season_games_dir) if f.endswith('.csv')]
        if game_files:
            latest_file = max(game_files, key=os.path.getmtime)
            df_games = pd.read_csv(latest_file, dtype={'GAME_ID': str})
            all_games.append(df_games)

if all_games:
    all_games_df = pd.concat(all_games, ignore_index=True)
    games_output_path = os.path.join(DATA_LAST_GAMES_MERGED_DIR, f"games_merged_all_seasons_{run_timestamp}.csv")
    
    os.makedirs(DATA_LAST_GAMES_MERGED_DIR, exist_ok=True)
    
    all_games_df.to_csv(games_output_path, index=False)
    print(f"✅ Merged games saved to {games_output_path} ({len(all_games_df)} rows)")


✅ Merged games saved to data\raw_last\games_merged\games_merged_all_seasons_2025-06-04_19-35-18.csv (35710 rows)


In [9]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-06-05 14:32:52.303039
Total time:  0:35:43.156881


In [10]:
print("\n✅ Scraping historique V3 terminé")



✅ Scraping historique V3 terminé
